## 学習手順

* 青いセルの説明を読む
* 白いセルに問題の解答を書く
* 黄色いセルを実行して確認する

セル内でしか使わない変数は、`_`で始まります。

## データについて

データは下記からダウンロードしたものを用意しています。

* https://www.kaggle.com/datasets/danbraswell/new-york-city-weather-18692022
* https://www.kaggle.com/datasets/pankajvermacool/titanic-traincsv
* https://www.kaggle.com/datasets/ahmadsamsulmuarif/online-retailcsv

## 準備

演習に必要なモジュールや演習のために用意したcolオブジェクトをインポートします。
colオブジェクトを使って、列名のExprを取得できます。たとえば、`col.Name`は`pl.col("Name")`と同じように使えます。

**このノートブックを開いたときは、最初に下記のセルを実行するようにしてください**。

In [3]:
from datetime import date
from textwrap import dedent
import polars as pl
import polars.selectors as cs
from study_polars2.col import col

_file = "../data/titanic_train.csv"
df_titanic = pl.read_csv(_file)
_file = "../data/NYC_Central_Park_weather_1869-2022.csv"
df_weather = pl.read_csv(_file, try_parse_dates=True)


---

### `問題 2.3.3` 文字列を含むか判定

1.1.3の`df_titanic`を次の条件で修正し、変数`ans`に代入してください。

* 列`Name`に`Mr.`が含まれているかどうかを取得する
  * 列名を`IsMr`とする

**解答欄**

In [129]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_titanic.with_columns(
    IsMr=col.Name.str.contains(r"Mr\.")
)
ans[:3]
```
</details>
<br>

**確認**

In [130]:
# このセルを実行してください
try:
    _ans = df_titanic.with_columns(
        IsMr=col.Name.str.contains(r"Mr\.")
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")

---

### `問題 2.3.10` カテゴリカル型へ変更

2.3.9の`df_titanic`を次の条件で修正し、変数`ans`に代入してください。

* 列`Pclass`を`1st`、`2nd`、`3rd`という値のカテゴリカル型に変換にする

最初の3行が下記のようになること。

| PassengerId | Survived | Pclass | ... |
| --: | --: | :-- | :-- |
| 1 | 0| "3rd" | ... |
| 2 | 1| "1st" | ... |
| 3 | 1| "3rd" | ... |

**解答欄**

In [94]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_titanic.with_columns(
    col.Pclass.replace_strict(
        {1: "1st", 2: "2nd", 3: "3rd"}
    ).cast(pl.Categorical)
)
ans[:3]
```

**別解*
```python
ans = df_titanic.with_columns(
    Pclass=pl.lit(["1st", "2nd", "3rd"])
    .list.get(col.Pclass - 1)
    .cast(pl.Categorical)
)
```
</details>
<br>

**確認**

In [92]:
# このセルを実行してください
try:
    _ans = df_titanic.with_columns(
        Pclass=pl.lit(["1st", "2nd", "3rd"])
        .list.get(col.Pclass - 1)
        .cast(pl.Categorical)
    )
    assert _ans.equals(ans) and ans["Pclass"].dtype == pl.Categorical
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")

---

### `問題 3.1.4` 数値列の集計

2.3.9の`df_titanic`を次の条件で修正し、変数`ans`に代入してください。

* 性別 (Sex) ごとに、すべての数値列の平均値を計算する
  * 列名のサフィックスに`Mean`をつける
* 列`Sex`で昇順にソートする

**解答欄**

In [168]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = (
    df_titanic.group_by("Sex")
    .agg(cs.numeric().mean().name.suffix("Mean"))
    .sort("Sex")
)
ans
```
</details>
<br>

**確認**

In [169]:
# このセルを実行してください
try:
    _ans = (
        df_titanic.group_by("Sex")
        .agg(cs.numeric().mean().name.suffix("Mean"))
        .sort("Sex")
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: cs.numeric()のような型セレクタを.agg()内で使用すると、対象となるすべての列に同じ集計関数が適用されます。
        .name.suffix()とすることで、サフィックスを付与できます。
    """))

---

### `問題 3.1.7` 条件を満たす行数

2.3.9の`df_titanic`を次の条件で修正し、変数`ans`に代入してください。

* 客室クラス (Pclass) ごとに、30歳以上の乗客の数を数える
  * 列名を`CountOver30`とする
* 列`Pclass`で昇順にソートする

**解答欄**

In [165]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = (
    df_titanic.group_by("Pclass")
    .agg(CountOver30=col.Age.filter(col.Age >= 30).len())
    .sort("Pclass")
)
ans[:3]
```

**別解**

```python
ans = (
    df_titanic.filter(col.Age >= 30)
    .group_by("Pclass").agg(CountOver30=pl.len())
    .sort("Pclass")
)
ans[:3]
```
</details>
<br>

**確認**

In [160]:
# このセルを実行してください
try:
    _ans = (
        df_titanic.group_by("Pclass")
        .agg(CountOver30=col.Age.filter(col.Age >= 30).len())
        .sort("Pclass")
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: 集計式の中で.filter()を使用することで、集計対象となる行をさらに絞り込むことができます。
        これは非常に強力な機能で、複雑な条件付き集計を簡潔に記述できます。
    """))

---

### `問題 3.2.1` 月ごとに集計

列`DATE`で昇順にソートされた`df_weather`を次の条件で修正し、変数`ans`に代入してください。

* 月ごとにグループ化し、各月の最高気温 (`TMAX`) の平均を計算する
  * 列名を`AvgTMAX`とする

最初の3行が下記のようになること。

| DATE       |   AvgTMAX |
| :--------- | --------: |
| 1869-01-01 | 39.516129 |
| 1869-02-01 | 39.714286 |
| 1869-03-01 |  41.83871 |

**解答欄**

In [186]:
_file = "../data/NYC_Central_Park_weather_1869-2022.csv"
df_weather = pl.read_csv(_file).with_columns(
    col.DATE.str.to_date()
)
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_weather.group_by_dynamic(
    index_column="DATE", every="1mo"
).agg(AvgTMAX=col.TMAX.mean())
ans[:3]
```
</details>
<br>

**確認**

In [187]:
# このセルを実行してください
try:
    _ans = df_weather.group_by_dynamic(
        index_column="DATE", every="1mo"
    ).agg(AvgTMAX=col.TMAX.mean())
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: .group_by_dynamic()のindex_columnに時間軸となる列を、everyにウィンドウの期間を指定します。
        "1mo"は1ヶ月、"1w"は1週間、"3d"は3日を意味します。
    """))

---

### `問題 3.2.3` 年ごとの集計(開始点変更)

3.2.1の`df_weather`を次の条件で修正し、変数`ans`に代入してください。

* 毎年7月を開始点として年間の降雪量(SNOW)の合計を計算する(例：2020/7/1 - 2021/6/30)
  * 列名を`TotalSnow`とする

**解答欄**

In [198]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_weather.group_by_dynamic(
    index_column="DATE",
    every="1y",
    offset="6mo",  # 7月を開始点にするため、6ヶ月オフセット
).agg(TotalSnow=col.SNOW.sum())
ans[:3]
```
</details>
<br>

**確認**

In [199]:
# このセルを実行してください
try:
    _ans = df_weather.group_by_dynamic(
        index_column="DATE",
        every="1y",
        offset="6mo",  # 7月を開始点にするため、6ヶ月オフセット
    ).agg(TotalSnow=col.SNOW.sum())
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")

---

### `問題 3.2.4` カテゴリカルキーと時間でグループ化

`_df`を次の条件で修正し、変数`ans`に代入してください。

* 地点(Station)ごと、かつ10年ごとに平均最高気温（TMAXの平均）を計算する
  * 列名を`AvgTMAX/decade`とする

**解答欄**

In [203]:
# ダミーの観測地点列を追加
_df = df_weather.with_columns(
    Station=pl.when(col.DATE.dt.year() < 1950)
    .then(pl.lit("StationA"))
    .otherwise(pl.lit("StationB"))
)
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = _df.group_by_dynamic(
    index_column="DATE",
    every="10y",  # 10年ごとに集計
    group_by="Station",  # カテゴリカルキーを指定
).agg(col.TMAX.mean().alias("AvgTMAX/decade"))
ans[:3]
```
</details>
<br>

**確認**

In [204]:
# このセルを実行してください
try:
    _ans = _df.group_by_dynamic(
        index_column="DATE",
        every="10y",  # 10年ごとに集計
        group_by="Station",  # カテゴリカルキーを指定
    ).agg(col.TMAX.mean().alias("AvgTMAX/decade"))
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: group_by_dynamicは、引数group_byを使って、時間ベースの動的グループ化と、カテゴリカルキーによる通常のグループ化を組み合わせることができます。
        これにより、例えば「観測地点ごと」に「月次」の集計を行うといった、より複雑な時系列分析が可能になります。
        Polarsはまずgroup_byで指定されたキーでデータを分割し、それぞれのグループ内で動的な時間ウィンドウ集計を効率的に実行します。
    """))

---

### `問題 3.2.5` ローリング集計

3.2.1の`df_weather`を次の条件で修正し、変数`ans`に代入してください。

* 7日間の移動平均最高気温を計算する
  * 列名を`AvgTMAX/7d`とする
* `DATE`と`AvgTMAX/7d`の2列とする

**解答欄**

In [219]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_weather.select(
    col.DATE,
    col.TMAX.rolling_mean(window_size=7).alias("AvgTMAX/7d"),
)
ans[5:8]
```
</details>
<br>

**確認**

In [220]:
# このセルを実行してください
try:
    _ans = df_weather.select(
        col.DATE,
        col.TMAX.rolling_mean(window_size=7).alias("AvgTMAX/7d"),
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: .group_by_dynamicはウィンドウを区切って集計するのに対し、.rolling_*系の関数は移動ウィンドウでの集計を行います。
    """))

---

### `問題 3.3.1` アップサンプリング

3.2.1の`df_weather`をダウンサンプリングした`_df`を次の条件で修正し、変数`ans`に代入してください。

  * 日次データとしてアップサンプリングする
  * 間のデータは線形補間する
  * 下記のようになること

| DATE       | AvgTMAX |
| :--------- | ------: |
| 1868-12-31 |    29.0 |
| 1869-01-01 |    30.0 |
| 1869-01-02 |    31.0 |
| 1869-01-03 |    35.5 |
| 1869-01-04 |    40.0 |

**解答欄**

In [223]:
_df = df_weather.group_by_dynamic(
    index_column="DATE", every="2d"
).agg(AvgTMAX=col.TMAX.mean())[:3]

# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = _df.upsample("DATE", every="1d").interpolate()
ans
```
</details>
<br>

**確認**

In [224]:
# このセルを実行してください
try:
    _ans = _df.upsample("DATE", every="1d").interpolate()
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: .interpolate()メソッドを使うと、null値が線形補間によって埋められます。
    """))

---

### `問題 3.4.8` Asof結合

時系列データで、各注文日に最も近い過去の市場価格を結合し、変数`ans`に代入してください。

**解答欄**

In [251]:
market_prices = pl.DataFrame({
    'time': pl.date_range(pl.date(2023, 1, 2), pl.date(2023, 1, 8), '2d', eager=True),
    'price': range(2, 9, 2),
})
orders = pl.DataFrame({
    'order_time': [date(2023, 1, 1), date(2023, 1, 5), date(2023, 1, 9)],
    'volume': [100, 120, 180],
})
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = orders.join_asof(
    market_prices,
    left_on="order_time",
    right_on="time",
    strategy="nearest",
)
ans
```
</details>
<br>

**確認**

In [252]:
# このセルを実行してください
try:
    _ans = orders.join_asof(
        market_prices,
        left_on="order_time",
        right_on="time",
        strategy="nearest",
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: join_asofは、キーが完全に一致しなくても、最も近いキーで結合する特殊な結合です。
        時系列データでイベントと直前の状態を紐付ける際などに非常に有用です。
        strategy引数で"backward"（過去方向、デフォルト）、"forward"（未来方向）、"nearest"（最も近い）を指定できます。
    """))

---

### `問題 3.4.9` 非等価結合

各イベントがどの時間枠に含まれるかを判定して結合し、event_timeでソートして、変数`ans`に代入してください。

**解答欄**

In [255]:
events = pl.DataFrame(
    {
        "event_time": [1.5, 2.8, 3.5, 5.1],
        "event_type": ["a", "b", "c", "d"],
    }
)
windows = pl.DataFrame(
    {
        "window_id": ["x", "y", "z"],
        "start_time": [1.0, 3.0, 5.0],
        "end_time": [2.0, 4.0, 6.0],
    }
)
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = events.join_where(
    windows,
    (pl.col("event_time") >= pl.col("start_time"))
    & (pl.col("event_time") < pl.col("end_time")),
).sort("event_time")
ans
```
</details>
<br>

**確認**

In [256]:
# このセルを実行してください
try:
    _ans = events.join_where(
        windows,
        (pl.col("event_time") >= pl.col("start_time"))
        & (pl.col("event_time") < pl.col("end_time")),
    ).sort("event_time")
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: join_whereは、等価条件(==)だけでなく、不等価条件(>, <, >=, <=)を含む任意の述語に基づいて結合を行うための実験的な機能です。
        これにより、クロス結合とフィルタリングを組み合わせるよりもはるかに効率的に、範囲ベースの結合（range join）などを実現できます。
    """))

---

### `問題 4.1.7` 自身を除くグループ内計算

4.1.1の`df_titanic`を次の条件で修正し、変数`ans`に代入してください。

* 各乗客について、その乗客を除く同じ客室クラスの他の乗客の平均運賃を計算する
  * 列名を`AvgFareOfOthersInClass`とする

**解答欄**

In [341]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_titanic.with_columns(
    AvgFareOfOthersInClass=(
        (col.Fare.sum().over("Pclass") - col.Fare)
        / (pl.len().over("Pclass") - 1)
    )
)
ans[:3]
```
</details>
<br>

**確認**

In [342]:
# このセルを実行してください
try:
    _ans = df_titanic.with_columns(
        AvgFareOfOthersInClass=(
            (col.Fare.sum().over("Pclass") - col.Fare)
            / (pl.len().over("Pclass") - 1)
        )
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")

---

### `問題 4.3.3` リスト内要素の変換

4.3.1の`df_word`を次の条件で修正し、変数`ans`に代入してください。

* 列`NameWords`の各単語をすべて大文字に変換する
* 列`NameWords`だけにする

**解答欄**

In [382]:
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = df_word.select(
    col.NameWords.list.eval(pl.element().str.to_uppercase())
)
ans[:3]
```
</details>
<br>

**確認**

In [383]:
# このセルを実行してください
try:
    _ans = df_word.select(
        col.NameWords.list.eval(pl.element().str.to_uppercase())
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: .list.eval()は、リスト内の各要素に対して式を適用するための強力なメソッドです。
        pl.element()はリスト内の個々の要素を参照します。
    """))

---

### `問題 4.3.6` JSONから構造体

`df_json`を次の条件で修正し、変数`df_struct`に代入してください。

* JSON文字列の列`ItemJson`を、構造体に変換する
  * 列名を`ItemStruct`とする
  * 構造体の型として、変数`dtype`の値を用いる

**解答欄**

In [400]:
df_json = pl.DataFrame(
    {
        "OrderID": [101, 102, 103],
        "ItemJson": [
            '{"name": "Laptop", "price": 1200}',
            '{"name": "Mouse", "price": 25}',
            '{"name": "Keyboard", "price": 75}',
        ],
    }
)
dtype = pl.Struct([
    pl.Field("name", pl.String),
    pl.Field("price", pl.Int64),
])

# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
df_struct = df_json.with_columns(
    ItemStruct=col.ItemJson.str.json_decode(dtype),
)
df_struct
```
</details>
<br>

**確認**

In [401]:
# このセルを実行してください
try:
    _ans = df_json.with_columns(
        ItemStruct=col.ItemJson.str.json_decode(dtype),
    )
    assert _ans.equals(df_struct)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: polars.Expr.str.json_decodeは、文字列型の列を効率的にパースし、pl.Struct 型の列に変換します。
        Structは、1つの列の中に複数のフィールド（この例では name と price）を持つことができる複合的なデータ型です。
        ans.schemaでスキーマを見ると、ItemStruct列がStruct型になっていることが確認できます。
    """))

---

### `問題 5.3.1` 折れ線グラフ

`_df`を使って次の要件で折れ線グラフを作成し、変数`ans`に代入してください。

* X軸は列`Name`
* Y軸は列`Score`
* Y軸のラベルは`得点`
* 列`Subject`ごとに線を引くこと

**解答欄**

In [457]:
import altair as alt

_df = pl.DataFrame({
    "Name": ["A", "A", "B", "B", "C", "C"],
    "Subject": ["Math", "Music", "Math", "Music", "Math", "Music"],
    "Score": [85, 90, 94, 88, 80, 93],
})
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = _df.plot.line(
    x="Name",
    y=alt.Y("Score", title="得点"),
    color="Subject",
)
ans
```
</details>
<br>

**確認**

In [458]:
# このセルを実行してください
try:
    _ans = _df.plot.line(
        x="Name",
        y=alt.Y("Score", title="得点"),
        color="Subject",
    )
    assert _ans.to_dict()["encoding"] == ans.to_dict()["encoding"]
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: 折れ線グラフは、plot.lineで、棒グラフはplot.barで描画できます。
    """))

---

### `問題 5.3.2` 散布図

`_df`を使って次の要件で散布図を作成し、変数`ans`に代入してください。

* X軸は列`Math`
* Y軸は列`Music`
* タイトルは`2教科の関係`

**解答欄**

In [499]:
import altair as alt

_df = pl.DataFrame({
    "Math": [33, 21, 42],
    "Music": [23, 34, 26],
})
# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
ans = _df.plot.scatter(
    x="Math",
    y="Music",
).properties(
    title="2教科の関係",
)
ans
```
</details>
<br>

**確認**

In [500]:
# このセルを実行してください
try:
    _ans = _df.plot.scatter(
        x="Math",
        y="Music",
    ).properties(
        title="2教科の関係",
    )
    assert _ans.to_dict()["encoding"] == ans.to_dict()["encoding"]
    assert _ans.to_dict()["title"] == ans.to_dict()["title"]
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")
    print(dedent("""\
        解説: 散布図は、plot.scatterで描画できます。
        これはaltair.Chartオブジェクトなので、propertiesで属性を設定できます。
    """))

---

### `問題 5.4.2` 顧客のRFM分析

この最終演習では、Online Retailデータセットを用いて、顧客セグメンテーションの一般的な手法であるRFM（Recency, Frequency, Monetary）分析を行います。これは、データクリーニング、時系列処理、複雑な集計など、多くのスキルを統合する実践的な課題です。

`_df`を元に、各顧客についてRFM指標を計算し、最終結果を変数`ans`に代入してください。

**手順**

1. データクリーニング
  * InvoiceDateをdatetime型に変換(`format="%m/%d/%y %H:%M"`)
  * Quantity > 0、UnitPrice > 0、CustomerIDが欠損以外でフィルタリング

2. RFM指標を計算
  * CustomerIDでグループ化し以下を修正
    * 列`Recency`として、「グループ化前のInvoiceDateの最大値」からInvoiceDateの最大値を引いた日数+1
    * 列`Frequency`として、InvoiceNoのユニーク数
    * 列`Monetary`として、(Quantity * UnitPrice)の合計

3. 列`Monetary`で降順に、列`CustomerID`で昇順にソート

最初の3行が下記のようになること。

| CustomerID | Recency | Frequency |  Monetary |
|-----------:|--------:|----------:|----------:|
|      14646 |       7 |        29 | 121973.65 |
|      18102 |      12 |        20 | 106601.55 |
|      12346 |     160 |         1 |   77183.6 |

**解答欄**

In [496]:
_file = "../data/online_retail.parquet"
_df = pl.scan_parquet(_file)

# ここから解答を作成してください


<details><summary>解答例</summary>
<br>

```python
# 1. データクリーニング
_tmp = _df.with_columns(
    col.InvoiceDate.str.to_datetime(format="%m/%d/%y %H:%M"),
).filter(
    col.Quantity > 0,
    col.UnitPrice > 0,
    col.CustomerID.is_not_null(),
)

# 2. RFM指標を計算
_tmp = (
    _tmp.with_columns(
        Recency=(
            col.InvoiceDate.max() - col.InvoiceDate.max().over("CustomerID")
        ).dt.total_days() + 1,
    )
    .group_by("CustomerID")
    .agg(
        col.Recency.first(),
        Frequency=col.InvoiceNo.n_unique(),
        Monetary=(col.Quantity * col.UnitPrice).sum(),
    )
)

# 3. 列`Monetary`で降順に、列`CustomerID`で昇順にソート
ans = _tmp.sort(["Monetary", "CustomerID"], descending=[True, False]).collect()

ans[:3]
```
</details>
<br>

**確認**

In [497]:
# このセルを実行してください
try:
    _ans = (
        pl.scan_parquet(_file).with_columns(
            col.InvoiceDate.str.to_datetime(format="%m/%d/%y %H:%M")
        ).filter(
            col.Quantity > 0, col.UnitPrice > 0, col.CustomerID.is_not_null()
        ).with_columns(
            Recency=(
                col.InvoiceDate.max() - col.InvoiceDate.max().over("CustomerID")
            ).dt.total_days() + 1,
        ).group_by("CustomerID").agg(
            col.Recency.first(),
            Frequency=col.InvoiceNo.n_unique(),
            Monetary=(col.Quantity * col.UnitPrice).sum(),
        ).sort(["Monetary", "CustomerID"], descending=[True, False]).collect()
    )
    assert _ans.equals(ans)
except (AssertionError, NameError):
    print("\x1b[31mNG\x1b[39m")
    print("解答例を確認してください")
else:
    print("\x1b[32mOK\x1b[39m")

---